# Extratropical Modes of Variability — Batch Pattern Maps

Iterates over **all modes × all seasons** to produce EOF pattern and teleconnection slope figures,
using `util.pcmdi_mov_reader.EMOVDiagReader` and `util.mov_plotter.ExtrapropicalModeMapPlotter`.

**Notebook structure:**
1. Imports (including `ExtrapropicalModeMapPlotter` from `util/`)
2. Shared configuration (paths, modes, models, reader + plotter)
3. Batch loop — all modes × all seasons

> For interactive single-mode exploration, see [`plot_mov_pattern.ipynb`](plot_mov_pattern.ipynb).

## Imports

In [ ]:
import os
from typing import Dict, Sequence, Optional, Tuple, Any, List

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math

import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from util.pcmdi_mov_reader import ModeFileSpec, EMOVDiagReader
from util.mov_plotter import ExtrapropicalModeMapPlotter

## Shared Configuration

Run this cell once before the batch loop. Defines paths, the mode registry, model list, and instantiates the reader and plotter.

In [ ]:
TOP_DIR  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_rrm_paper"
DATA_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi/overview_rrm"
FIG_DIR  = "./"
os.makedirs(FIG_DIR, exist_ok=True)

# Mode configuration: EOF component, driving variable, obs dataset, and optional
# regional bounds (lat_bnds/lon_bnds) used for pattern plots and stats.
mode_dict = {
    # SST-based ocean modes
    "AMO":  {"eof": "eof1", "var": "ts",  "obs": "HadISST2",  "lat_bnds": (0, 70),    "lon_bnds": (-80, 0)},
    "PDO":  {"eof": "eof1", "var": "ts",  "obs": "HadISST2"},
    "NPGO": {"eof": "eof2", "var": "ts",  "obs": "HadISST2"},
    # Northern Hemisphere SLP modes
    "NAM":  {"eof": "eof1", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (20, 90),   "lon_bnds": (-180, 180)},
    "NAO":  {"eof": "eof1", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (20, 80),   "lon_bnds": (-80, 40)},
    "NPO":  {"eof": "eof2", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (20, 80),   "lon_bnds": (120, -120)},  # lon wrap
    "PNA":  {"eof": "eof1", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (20, 80),   "lon_bnds": (120, -60)},
    # Southern Hemisphere SLP modes
    "SAM":  {"eof": "eof1", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (-90, -20), "lon_bnds": (-180, 180)},
    "PSA1": {"eof": "eof2", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (-90, -20), "lon_bnds": (120, -60)},
    "PSA2": {"eof": "eof3", "var": "psl", "obs": "NOAA-20C", "lat_bnds": (-90, -20), "lon_bnds": (120, -60)},
}

MODEL = [
    "v3.LR.amip_0101",
    "v3.NARRM.amip_0101",
    "v3.NARRM_r0125.amip_0101",
    "v3.EARRM.amip_0101",
    "v3.AMZRRM.amip_0101",
]
PERIOD = (1985, 2014)

obs_dataset_map = {m: cfg["obs"] for m, cfg in mode_dict.items()}

reader = EMOVDiagReader(
    data_dir=DATA_DIR,
    prefer_cbf=True,
    obs_key="obs",
    model_key="e3sm",
    target_lat_name="latitude",
    target_lon_name="longitude_a",
    obs_dataset_map=obs_dataset_map,
    normalize_lon_360=True,
    sort_latlon=True,
)

plotter = ExtrapropicalModeMapPlotter(
    fig_dir=FIG_DIR,
    plot_dict={"reference": {"label": "Obs"}, "hist": {"label": "E3SM"}},
    group_order=("hist",),
    obs_key="reference",
    lat_name="latitude",
    lon_name="longitude_a",
)

## Batch Loop: All Modes × All Seasons

Iterates over every mode and its applicable season/frequency tokens.
Uncomment the `plotter.plot_mode_season_maps(...)` call to write figures to disk.

In [ ]:
# ── Batch settings ────────────────────────────────────────────────────────────
SEASONS_PSL = ["DJF", "MAM", "JJA", "SON", "monthly", "yearly"]
FREQ_TS     = "monthly"   # token for ts-based modes; change to "yearly" if needed
# ──────────────────────────────────────────────────────────────────────────────

for mode, cfg in mode_dict.items():
    var    = cfg["var"]
    eof    = cfg["eof"]
    tokens = SEASONS_PSL if var == "psl" else [FREQ_TS]

    for token in tokens:
        spec = ModeFileSpec(mode=mode, var=var, eof=eof, period=PERIOD, season_or_freq=token)

        dd_pattern = reader.build_multimodel_stack(MODEL, spec, which="eof",   member_dim="member")
        dd_slope   = reader.build_multimodel_stack(MODEL, spec, which="slope", member_dim="member")

        products = {
            "pattern":  dd_pattern,
            "teleconn": dd_slope,
        }
        product_labels = {
            "pattern":  f"{mode} {token} ({var}, {eof.upper()}) EOF pattern",
            "teleconn": f"{mode} {token} ({var}, {eof.upper()}) teleconn slope",
        }
        cb_labels = {"pattern": "EOF pattern", "teleconn": "Regression slope"}
        fname     = f"{mode}_{token}_{var}_{eof}_pattern_teleconn.pdf"

        # Uncomment to produce figures:
        # plotter.plot_mode_season_maps(
        #     mode=mode,
        #     season=token,
        #     products=products,
        #     product_order=["pattern", "teleconn"],
        #     product_labels=product_labels,
        #     cb_labels_by_product=cb_labels,
        #     filename=fname,
        #     overlay_spread=True,
        #     spread_quantile=0.75,
        #     cmap="RdBu_r",
        #     one_colorbar_per_row=True,
        # )
        print("Prepared:", fname)